In [1]:
#MCLP Implementation to determine optimal healthcare facility locations in a city

In [2]:
!pip install ortools
!pip install haversine

In [3]:
#Importing Libraries 
from ortools.linear_solver import pywraplp
import numpy as np
import math 
import statistics
import pandas as pd
import haversine as hs
import matplotlib.pyplot as plt 
import plotly.express as px

In [4]:
#Reading File
df=pd.read_csv("Philadelphia ZCTAs Sub Service line level_Cardiology_Medical Cardiology.csv")
df

,zcta,service_line,subservice_line,base_year_demand_volume,five_year_demand_volume,ten_year_demand_volume,weighted demand,internal_pt_lat,internal_pt_lon
0,8001,Cardiology,Medical Cardiology,161.877470,161.990214,160.314657,161.464998,39.547880,-75.346384
1,8002,Cardiology,Medical Cardiology,4836.830019,4836.604184,4781.138736,4820.009717,39.931978,-75.027557
2,8003,Cardiology,Medical Cardiology,6576.749261,6575.578988,6498.732818,6552.759192,39.882703,-74.972036
3,8004,Cardiology,Medical Cardiology,2364.391262,2367.632443,2346.182027,2360.549082,39.764747,-74.870626
4,8007,Cardiology,Medical Cardiology,1121.537593,1121.571846,1108.864051,1117.752657,39.865790,-75.053745
...,...,...,...,...,...,...,...,...,...
502,21918,Cardiology,Medical Cardiology,862.130899,863.336241,855.516557,860.749268,39.678194,-76.175888
503,21919,Cardiology,Medical Cardiology,739.079538,739.110158,730.725531,736.588646,39.418554,-75.933709
504,21920,Cardiology,Medical Cardiology,52.342622,52.382513,51.849664,52.214680,39.657498,-75.828326
505,21921,Cardiology,Medical Cardiology,8830.584994,8838.890462,8751.196429,8810.921159,39.627145,-75.859891


In [5]:
#Standard Data conversion
X = df['weighted demand']
X = X.to_numpy()
#X

In [6]:
#Standard Data conversion
Y = df[['internal_pt_lat','internal_pt_lon']]
Y = Y.to_numpy()
#Y

In [7]:
#Intializing some constraints and variables
min_demand=5000;                                                                #min demand at a facility that is needed for it to remain profitable. 
                                                                                 #This could be derived from financial modelling of the demand in the market.
total_demand = df['weighted demand'].sum()
max_travel_time=15;                                                              #max travel time
max_distance=((max_travel_time/60)*30);                                          #max distance patient is willing to travel
rows=len(df);
cols=len(df);
c=[]                                                                             #For data visualization
decision_data=[];                                                                #For data visualization
distances=[[0 for x in range(cols)] for x in range(rows)];
pfac = {};

X_var = {}
Y_var = [None for _ in range(cols)]
eligible = [[] for _ in range(rows)]
incoming = [[] for _ in range(cols)]
sorted_sites = [[] for _ in range(rows)]
sorted_sites_active = [[] for _ in range(rows)]
max_possible = [0.0 for _ in range(cols)]
active_facilities = []
active_set = set()
eligible_active = [[] for _ in range(rows)]
incoming_active = [[] for _ in range(cols)]

X_decision=[-1 for x in range(cols)];                                            #For data visualization
Demand_dist=[0 for x in range(cols)];                                            #For data visualization


In [8]:
#Calculating inter location distances
for i in range(rows):
    for j in range(cols):
        loc1 = Y[i]
        loc2 = Y[j]
        distances[i][j] = (hs.haversine(loc1, loc2)) / 1.609
        if distances[i][j] <= max_distance:
            eligible[i].append(j)
            incoming[j].append(i)

for i in range(rows):
    sorted_sites[i] = sorted(range(cols), key=lambda jj: distances[i][jj])


In [9]:
# Participation factor calculation
for i in range(rows):
    for j in eligible[i]:
        pfac[i, j] = X[i] * (1 - (distances[i][j] / max_distance))

for j in range(cols):
    max_possible[j] = sum(pfac[i, j] for i in incoming[j])

active_facilities = [j for j in range(cols) if max_possible[j] >= min_demand]
active_set = set(active_facilities)

for i in range(rows):
    eligible_active[i] = [j for j in eligible[i] if j in active_set]
    sorted_sites_active[i] = [j for j in sorted_sites[i] if j in active_set]

for j in active_facilities:
    incoming_active[j] = incoming[j][:]

print('Facilities screened out by min_demand upper bound =', cols - len(active_facilities))


Facilities screened out by min_demand upper bound = 120


In [ ]:
#Main function (Implemented exactly as per (https://ideas.repec.org/a/spr/annopr/v110y2002i1p123-13210.1023-a1020767501233.html) paper)
def main():
    # Create the mip solver with the SCIP backend.
    solver = pywraplp.Solver.CreateSolver('SCIP')
    #if not solver:
    #    return    
    
    # Variable declaration
    for i in range(rows):
        for j in eligible_active[i]:
            X_var[i, j] = solver.BoolVar(f'x{i}_{j}')

    for j in active_facilities:
        Y_var[j] = solver.BoolVar(f'y{j}')

    print('Number of variables =', solver.NumVariables())

    
#Constraints
     
    
#No. of facilites constraint to limit solution space(use for A/B tests)
    # demand=20000;
    # total_demand = df['weighted demand'].sum()
    # number_of_facilites=(int)(total_demand/demand)                                   
    # num_fac=Y_var[0]
    # for i in range(1,cols):
    #       num_fac=num_fac + Y_var[i];
    
    # solver.Add(num_fac<= number_of_facilites)
    
    
#Only one facility assignment constraint
    for i in range(rows):
        solver.Add(solver.Sum(X_var[i, j] for j in eligible_active[i]) <= 1)

    
    
#Y variable greater than equal to X constraint
    for i in range(rows):
        for j in eligible_active[i]:
            solver.Add(X_var[i, j] <= Y_var[j])

# Min demand constraint
    for j in active_facilities:
        solver.Add(
            solver.Sum(pfac[i, j] * X_var[i, j] for i in incoming_active[j]) >= min_demand * Y_var[j]
        )

        
    
 #   print('Number of constraints =', solver.NumConstraints())
    
#Objective function formulation  
    solver.Maximize(
        solver.Sum(pfac[i, j] * X_var[i, j] for i in range(rows) for j in eligible_active[i])
    )

    


    # ------------------------------------------------------------
    # Section 4 (Procedure I): Iteratively add violated "closest open facility" cuts
    # ------------------------------------------------------------
    EPS = 1e-9
    MAX_ITERS = min(200, max(20, cols))
    added_cuts = set()  # avoid duplicating the same constraint
    
    for it in range(MAX_ITERS):
        status = solver.Solve()
        if status != pywraplp.Solver.OPTIMAL:
            break
    
        # facilities open in the CURRENT solution
        open_facilities = [j for j in active_facilities if Y_var[j].solution_value() > 0.5]
        open_set = set(open_facilities)
        if not open_facilities:
            print("No open facilities found in solution. Stopping Procedure I loop.")
            break
    
        new_cuts = 0
        violations = 0
    
        for i in range(rows):
    
            # Find assigned facility for demand point i (<= 1 assignment)
            j_assigned = None
            for j in eligible_active[i]:
                if X_var[i, j].solution_value() > 0.5:
                    j_assigned = j
                    break                        
            if j_assigned is None:
                continue
    
            # Find the closest OPEN facility to i using the precomputed distance ordering
            j_closest_open = next(jj for jj in sorted_sites_active[i] if jj in open_set)

            # Violation if assigned facility is not the closest open facility
            if distances[i][j_closest_open] + EPS < distances[i][j_assigned]:
                violations += 1
                k = j_closest_open  # closest open facility
            
                # Only need to cut OPEN facilities (closed ones can't be assigned anyway)
                for j in open_facilities:
                    if j in eligible_active[i] and j != k and distances[i][k] + EPS < distances[i][j]:
                        cut_key = (i, j, k)
                        if cut_key not in added_cuts:
                            solver.Add(X_var[i, j] <= 1 - Y_var[k])
                            added_cuts.add(cut_key)
                            new_cuts += 1
        
        
        print(f"Procedure I iter {it+1}: open={len(open_facilities)}, violations={violations}, new_cuts={new_cuts}")
    
        # Converged when no new violated closest-open constraints are found
        if new_cuts == 0:
            break
    
    # After Procedure I loop, 'status' is from the last solve.
    
    
                 
    print('Number of constraints =', solver.NumConstraints())  
    

    if status == pywraplp.Solver.OPTIMAL:
        print('Solution:')
        print('Objective value =', solver.Objective().Value())

        
#Code to get decisions regarding which facilities should be opened         
        for j in range(cols):
            decision_data.append(Y_var[j].solution_value() if Y_var[j] is not None else 0.0)

#Code to generate visualization regarding which facilities should be opened             
        for j in active_facilities:
            if Y_var[j].solution_value() == 1.0:
                c.append(j)
        
 #       for i in range(rows):
 #           for j in eligible_active[i]:
 #                print('X'+str(i)+str(j), X_var[i, j].solution_value())

 #Code to get demand distribution      
        for i in range(rows):
            for j in eligible_active[i]:
                if X_var[i, j].solution_value() > 0.5:
                    X_decision[i] = j
    else:
        print('The problem does not have an optimal solution.')

    print('\nAdvanced usage:')
    print('Problem solved in %f milliseconds' % solver.wall_time())
    print('Problem solved in %d iterations' % solver.iterations())
    print('Problem solved in %d branch-and-bound nodes' % solver.nodes())

if __name__ == '__main__':
    main()

Number of variables = 9964
Procedure I iter 1: open=180, violations=1, new_cuts=7


In [ ]:
# solver = pywraplp.Solver.CreateSolver('GUROBI')
# if not solver:
#     raise RuntimeError("Gurobi backend is not available. Check installation and license.")


In [ ]:
#Scatter plot of facility locations to be opened against lat and long
y_points = df['internal_pt_lat']
x_points = df['internal_pt_lon']

plt.figure()
plt.grid(True)       
plt.scatter(x_points, y_points) 

for i in range(len(c)):
    plt.text(x_points[c[i]], y_points[c[i]], f'{c[i]}')
    

In [ ]:
#Code to generate optimal locations distribution visualization

for i in range(len(decision_data)):
   if decision_data[i]==0.0:
      decision_data[i]="no"
   if decision_data[i]==1.0:
      decision_data[i]="yes"
fig = px.scatter_geo(df,lat='internal_pt_lat',lon='internal_pt_lon',color=decision_data, hover_name="zcta",scope='usa', )
fig.update_layout(title = 'Optimal Locations', title_x=0.5)


fig.show()

In [ ]:
#X_decision

In [ ]:
#Code to calculate demand capture 
for i in range(rows):
    if X_decision[i]!=-1:
        Demand_dist[X_decision[i]]=Demand_dist[X_decision[i]]+ df.at[i,'weighted demand']
        
    #Demand_dist[i]=Demand_dist[i]+df[df["zcta"] == X_decision[i]]
for i in range(rows):
    Demand_dist[i]=(Demand_dist[i]/total_demand)*100

In [ ]:
#Code to generate demand capture table
print("Demand","    ","Zcta no.")
for i in range(rows):
    if(Demand_dist[i]>0):
       print(round(Demand_dist[i],1),"%","     ",df.at[i,'zcta'])

In [ ]:
#Code to generate demand capture by optimal locations
for i in range(len(Demand_dist)):
    Demand_dist[i]=round(Demand_dist[i],1)

for i in range(len(decision_data)):
   if decision_data[i]==0.0:
      decision_data[i]="no"
   if decision_data[i]==1.0:
      decision_data[i]="yes"
fig = px.scatter_geo(df,lat='internal_pt_lat',lon='internal_pt_lon',color=decision_data, hover_name=Demand_dist,scope='usa' )
fig.update_layout(title = 'Demand Capture', title_x=0.5)


fig.show()

In [ ]:
#The red locations are the locations which algorithms predicts to be the optimal locations to open healthcare facility 